# DIP-Pipeline: Kontextvalidierung & Rohdaten-Ingestion

Themenfilter: `Rente`, `Altersabsicherung`

Benötigte Pakete: `requests`, `pyyaml` (Standardbibliothek: `sqlite3`, `hashlib`, `json`, `os`, `time`, `datetime`)

Vor dem Ausführen muss die Umgebungsvariable `DIP_API_KEY` gesetzt sein.

**Zwei API-Einschränkungen, die die Abfrage-Logik unten prägen:**
- `/vorgang` bietet keinen Volltext-/Stichwort-Parameter → Keyword-Filterung erfolgt client-seitig auf `titel`/`abstract`.
- `/drucksache` und `/plenarprotokoll` haben keinen `f.vorgang`-Filter → die zugehörigen Dokumente werden über `fundstelle` der Vorgangspositionen ermittelt und gebündelt per `f.id` nachgeladen.

## 0. Setup

In [26]:
import hashlib
import json
import os
import sqlite3
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests
import yaml

DIP_BASE_URL = "https://search.dip.bundestag.de/api/v1"
DIP_OPENAPI_URL = "https://search.dip.bundestag.de/api/v1/openapi.yaml"
DIP_TERMS_URL = "https://dip.bundestag.de/documents/nutzungsbedingungen_dip.pdf"

API_KEY = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"

DB_PATH = Path("dip_rohdaten.sqlite")

SEARCH_TERMS = ["Rente", "Altersabsicherung", "Altersvorsorge", "Altersarmut", "Rentenversicherung", "Rentenreform", "Rentenanpassung"]
UPDATE_BUFFER_MINUTES = 15
INITIAL_TIMEFRAME_DAYS = 30

**Datenbankschema:** je eine Rohdaten-Tabelle pro Ressourcentyp (`id` als Primärschlüssel, `aktualisiert` für den Update-Abgleich, `raw_json` als vollständiger API-Response – Bereinigung/Minimierung passiert erst später im dbt-Modelling). Dazu drei Betriebstabellen: `kontext_validierung_log` (Protokoll der Kontextchecks), `kontext_referenzwerte` (letzter bekannter Hash/Snapshot je Check) und `sync_state` (Zeitpunkt der letzten erfolgreichen Abfrage je Ressourcentyp).

In [27]:
def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode = WAL;")
    return conn


def init_db():
    conn = get_connection()
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS vorgang (
            id TEXT PRIMARY KEY,
            aktualisiert TEXT NOT NULL,
            titel TEXT,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS vorgangsposition (
            id TEXT PRIMARY KEY,
            vorgang_id TEXT NOT NULL,
            aktualisiert TEXT NOT NULL,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS drucksache (
            id TEXT PRIMARY KEY,
            aktualisiert TEXT NOT NULL,
            dokumentnummer TEXT,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS plenarprotokoll (
            id TEXT PRIMARY KEY,
            aktualisiert TEXT NOT NULL,
            dokumentnummer TEXT,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS kontext_validierung_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            zeitpunkt TEXT NOT NULL,
            pruefung TEXT NOT NULL,
            status TEXT NOT NULL,
            details TEXT
        );

        CREATE TABLE IF NOT EXISTS kontext_referenzwerte (
            schluessel TEXT PRIMARY KEY,
            wert TEXT NOT NULL,
            gesetzt_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS sync_state (
            ressourcentyp TEXT PRIMARY KEY,
            letzte_abfrage TEXT NOT NULL
        );
        """
    )
    conn.commit()
    conn.close()


init_db()

## 1. Kontextvalidierung

Drei unabhängige, deterministische Vor-Checks, die vor jedem Pipeline-Lauf ausgeführt werden. Jeder Check schreibt sein Ergebnis in `kontext_validierung_log` und aktualisiert seinen Referenzwert in `kontext_referenzwerte`. Status `geaendert` oder `fehler` soll den eigentlichen Datenabruf blockieren (siehe Abschnitt 3).

**OpenAPI-Schema-Check** – lädt die aktuelle OpenAPI-Spezifikation, vergleicht Endpunktliste und Schema-Hashes gegen den letzten bekannten Snapshot und protokolliert konkrete Änderungen (neue/entfernte endpoints, neue/geänderte Schemas).

In [28]:
def openapi_schema_check(conn):
    response = requests.get(DIP_OPENAPI_URL, timeout=30)
    response.raise_for_status()
    schema = yaml.safe_load(response.text)

    endpoints = sorted(schema.get("paths", {}).keys())
    schema_hashes = {
        name: hashlib.sha256(json.dumps(definition, sort_keys=True).encode()).hexdigest()
        for name, definition in schema.get("components", {}).get("schemas", {}).items()
    }

    referenz_zeile = conn.execute(
        "SELECT wert FROM kontext_referenzwerte WHERE schluessel = 'openapi_snapshot'"
    ).fetchone()

    changes = []
    if referenz_zeile is not None:
        referenz = json.loads(referenz_zeile[0])
        new_endpoints = sorted(set(endpoints) - set(referenz["endpoints"]))
        deleted_endpoints = sorted(set(referenz["endpoints"]) - set(endpoints))
        new_schemas = sorted(set(schema_hashes) - set(referenz["schema_hashes"]))
        changed_schemas = sorted(
            name
            for name, wert in schema_hashes.items()
            if name in referenz["schema_hashes"] and referenz["schema_hashes"][name] != wert
        )

        if new_endpoints:
            changes.append(f"neue endpoints: {new_endpoints}")
        if deleted_endpoints:
            changes.append(f"entfernte endpoints: {deleted_endpoints}")
        if new_schemas:
            changes.append(f"neue Schemas: {new_schemas}")
        if changed_schemas:
            changes.append(f"geänderte Schemas: {changed_schemas}")

    status = "geaendert" if changes else "ok"
    details = "; ".join(changes) if changes else "keine strukturelle Änderung festgestellt"

    conn.execute(
        "INSERT INTO kontext_validierung_log (zeitpunkt, pruefung, status, details) VALUES (?, 'openapi_schema', ?, ?)",
        (datetime.now(timezone.utc).isoformat(), status, details),
    )
    conn.execute(
        """
        INSERT OR REPLACE INTO kontext_referenzwerte (schluessel, wert, gesetzt_am)
        VALUES ('openapi_snapshot', ?, ?)
        """,
        (
            json.dumps({"endpoints": endpoints, "schema_hashes": schema_hashes}),
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    conn.commit()
    return status

**API-Key-Check** – führt eine minimale, echte Testanfrage aus (statt Website-Scraping) und wertet den HTTP-Statuscode aus: `200` = gültig, `401` = Key wurde offenbar geändert/ist ungültig.

In [29]:
def api_key_check(conn):
    try:
        response = requests.get(
            f"{DIP_BASE_URL}/vorgang",
            params={"apikey": API_KEY, "f.id": 1},
            timeout=15,
        )
        if response.status_code == 200:
            status, details = "ok", "API-Key gültig"
        elif response.status_code == 401:
            status, details = "geaendert", "API-Key wird abgelehnt (401) - vermutlich geändert/abgelaufen"
        else:
            status, details = "fehler", f"unerwarteter Statuscode {response.status_code} beim Key-Check"
    except requests.RequestException as fehler:
        status, details = "fehler", f"Netzwerkfehler beim Key-Check: {fehler}"

    conn.execute(
        "INSERT INTO kontext_validierung_log (zeitpunkt, pruefung, status, details) VALUES (?, 'api_key', ?, ?)",
        (datetime.now(timezone.utc).isoformat(), status, details),
    )
    conn.commit()
    return status

**Nutzungsbedingungen-Check** – lädt das PDF der Nutzungsbedingungen, hasht den Inhalt und vergleicht gegen den letzten bekannten Hash.

In [30]:
def terms_check(conn):
    response = requests.get(DIP_TERMS_URL, timeout=30)
    response.raise_for_status()
    current_hash = hashlib.sha256(response.content).hexdigest()

    referenz = conn.execute(
        "SELECT wert FROM kontext_referenzwerte WHERE schluessel = 'nutzungsbedingungen_hash'"
    ).fetchone()

    if referenz is None:
        status, details = "ok", "Erststart: Referenzwert gesetzt, kein Vergleich möglich"
    elif referenz[0] == current_hash:
        status, details = "ok", "keine Änderung der Nutzungsbedingungen festgestellt"
    else:
        status, details = "geaendert", "Nutzungsbedingungen-PDF hat sich verändert - manuelle Prüfung nötig"

    conn.execute(
        """
        INSERT INTO kontext_validierung_log (zeitpunkt, pruefung, status, details)
        VALUES (?, 'nutzungsbedingungen', ?, ?)
        """,
        (datetime.now(timezone.utc).isoformat(), status, details),
    )
    conn.execute(
        """
        INSERT OR REPLACE INTO kontext_referenzwerte (schluessel, wert, gesetzt_am)
        VALUES ('nutzungsbedingungen_hash', ?, ?)
        """,
        (current_hash, datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()
    return status

**Sammelfunktion** – führt alle drei Checks aus und bricht mit einer Exception ab, falls mindestens einer `geaendert` oder `fehler` meldet. Details stehen in `kontext_validierung_log`.

In [31]:
def validate_context():
    conn = get_connection()
    results = {
        "openapi_schema": openapi_schema_check(conn),
        "api_key": api_key_check(conn),
        "nutzungsbedingungen": terms_check(conn),
    }
    conn.close()

    critical = [name for name, status in results.items() if status in {"geaendert", "fehler"}]
    if critical:
        raise RuntimeError(
            f"Kontextvalidierung fehlgeschlagen für: {critical} - Details siehe kontext_validierung_log"
        )
    return results

## 2. API-Request und Rohdaten-Speicherung

**Generische Abfragefunktion** – kapselt Cursor-Pagination (Folgeanfragen bis der Cursor sich nicht mehr ändert) sowie Retry mit exponentiellem Backoff bei `429`/`5xx`. `params` kann Listenwerte enthalten (z.B. `f.id`), `requests` wiederholt den Parameter dann automatisch.

In [32]:
def _request_mit_retry(url, params, max_versuche=5):
    cool_off = 1.0
    for versuch in range(1, max_versuche + 1):
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 200:
            return response
        if response.status_code in (429, 500, 502, 503, 504) and versuch < max_versuche:
            time.sleep(cool_off)
            cool_off *= 2
            continue
        response.raise_for_status()
    raise RuntimeError(f"Anfrage an {url} nach {max_versuche} Versuchen fehlgeschlagen")


def dip_request(ressourcentyp, params):
    basis_params = {"apikey": API_KEY, **params}
    all_docs = []
    cursor = None

    while True:
        request_params = dict(basis_params)
        if cursor is not None:
            request_params["cursor"] = cursor

        response = _request_mit_retry(f"{DIP_BASE_URL}/{ressourcentyp}", request_params)
        data = response.json()
        all_docs.extend(data.get("documents", []))

        new_cursor = data.get("cursor")
        if new_cursor is None or new_cursor == cursor:
            break
        cursor = new_cursor
        time.sleep(0.2)

    return all_docs

**Sync-State-Hilfsfunktionen** – lesen/schreiben den Zeitpunkt der letzten erfolgreichen Abfrage je Ressourcentyp; Basis für das `f.aktualisiert.start`-Fenster inkl. 15-Minuten-Überlappungspuffer.

In [33]:
def get_last_request(conn, ressource_type, standard_time):
    line = conn.execute(
        "SELECT letzte_abfrage FROM sync_state WHERE ressourcentyp = ?", (ressource_type,)
    ).fetchone()
    return line[0] if line else standard_time


def set_last_request(conn, ressource_type, time):
    conn.execute(
        "INSERT OR REPLACE INTO sync_state (ressourcentyp, letzte_abfrage) VALUES (?, ?)",
        (ressource_type, time),
    )
    conn.commit()

**Vorgänge abfragen** – zieht Vorgänge über `f.aktualisiert.start` (initial: letzte 30 Tage, danach: seit letzter Abfrage minus Puffer), filtert client-seitig auf die SEARCH_TERMS in `titel`/`abstract` und speichert Treffer per Upsert (`INSERT OR REPLACE`) in SQLite.

In [34]:
def includes_search_terms(procedure):
    text = " ".join(filter(None, [procedure.get("titel"), procedure.get("abstract")])).lower()
    return any(term.lower() in text for term in SEARCH_TERMS)


def request_procedures(conn):
    run_start = datetime.now(timezone.utc)
    standard_start = (run_start - timedelta(days=INITIAL_TIMEFRAME_DAYS)).isoformat()
    last_request = get_last_request(conn, "vorgang", standard_start)
    request_start = (
        datetime.fromisoformat(last_request) - timedelta(minutes=UPDATE_BUFFER_MINUTES)
    ).isoformat()

    raw_data = dip_request("vorgang", {"f.aktualisiert.start": request_start})
    relevant_procedures = [procedure for procedure in raw_data if includes_search_terms(procedure)]

    return relevant_procedures, run_start

def save_procedures(conn, procedures):
    for procedure in procedures:
        conn.execute(
            """
            INSERT OR REPLACE INTO vorgang (id, aktualisiert, titel, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                procedure["id"],
                procedure["aktualisiert"],
                procedure.get("titel"),
                procedure.get("datum"),
                json.dumps(procedure, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )
    conn.commit()

**Kinddaten je Vorgang** – holt die Vorgangspositionen per `f.vorgang=<id>`, sammelt daraus die `fundstelle`-IDs (getrennt nach `Drucksache`/`Plenarprotokoll`) und lädt die zugehörigen Dokumente gebündelt per wiederholtem `f.id`-Parameter nach. Läuft für jeden in diesem Durchlauf gefundenen bzw. aktualisierten Vorgang.

In [35]:
def _docs_by_id_batch(ressource_typ, ids, batch_size=50):
    ids = sorted(ids)
    docs = []
    for start in range(0, len(ids), batch_size):
        batch = ids[start : start + batch_size]
        docs.extend(dip_request(ressource_typ, {"f.id": batch}))
    return docs


def request_child_data(procedure_id):
    procedure_positions = dip_request("vorgangsposition", {"f.vorgang": procedure_id})
    source_ids = {"Drucksache": set(), "Plenarprotokoll": set()}

    for vp in procedure_positions:
        source = vp.get("fundstelle")
        if source:
            source_ids[source["dokumentart"]].add(source["id"])

    documents = _docs_by_id_batch("drucksache", source_ids["Drucksache"])
    protocols = _docs_by_id_batch("plenarprotokoll", source_ids["Plenarprotokoll"])

    return {
        "vorgangsposition": procedure_positions,
        "drucksache": documents,
        "plenarprotokoll": protocols,
    }


def save_child_data(conn, child_data):
    for vp in child_data["vorgangsposition"]:
        conn.execute(
            """
            INSERT OR REPLACE INTO vorgangsposition (id, vorgang_id, aktualisiert, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                vp["id"],
                vp["vorgang_id"],
                vp["aktualisiert"],
                vp.get("datum"),
                json.dumps(vp, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )

    for document in child_data["drucksache"]:
        conn.execute(
            """
            INSERT OR REPLACE INTO drucksache (id, aktualisiert, dokumentnummer, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                document["id"],
                document["aktualisiert"],
                document.get("dokumentnummer"),
                document.get("datum"),
                json.dumps(document, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )

    for protocol in child_data["plenarprotokoll"]:
        conn.execute(
            """
            INSERT OR REPLACE INTO plenarprotokoll (id, aktualisiert, dokumentnummer, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                protocol["id"],
                protocol["aktualisiert"],
                protocol.get("dokumentnummer"),
                protocol.get("datum"),
                json.dumps(protocol, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )

    conn.commit()

**Orchestrierung** – verbindet beide Schritte: erst die (inkrementell gefilterten) Vorgänge laden, dann für jeden davon die Kinddaten nachziehen.

In [ ]:
def sync_procedures_and_child_data():
    conn = get_connection()
    procedures, run_start = request_procedures(conn)
    save_procedures(conn, procedures)
    set_last_request(conn, "vorgang", run_start.isoformat())

    for procedure in procedures:
        child_data = request_child_data(procedure["id"])
        save_child_data(conn, child_data)

    conn.close()
    return len(procedures)

## 3. Pipeline-Einstiegspunkt (für den Scheduler)

Kontextvalidierung läuft vor jedem Datenabruf und blockiert diesen bei kritischem Ergebnis. Diese Funktion ist der Einstiegspunkt, den ein Scheduler (z.B. APScheduler oder cron + papermill) periodisch aufruft.

In [37]:
def pipeline_run():
    validate_context()
    count_procedures = sync_procedures_and_child_data()
    return count_procedures


pipeline_run()

KeyboardInterrupt: 